In [1]:
import pandas as pd

In [2]:
import google.colab
google.colab.drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/CS1090B/project/data/'


Mounted at /content/drive


In [5]:
df = pd.read_parquet(DATA_ROOT + 'all_trades_features.parquet')
# 1. Ensure data is sorted chronologically (critical for time-series)
df = df.sort_values('created_time').reset_index(drop=True)

# 2. Calculate the integer index for the 80% mark
split_idx = int(len(df) * 0.8)

# 3. Get the exact timestamp at that index
split_date = df['created_time'].iloc[split_idx]

print(f"Total rows: {len(df):,}")
print(f"80% split index: {split_idx:,}")
print(f"Date at 80% mark: {split_date}")

Total rows: 63,801,798
80% split index: 51,041,438
Date at 80% mark: 2025-11-06 03:58:08.028990+00:00


In [ ]:
import os, gc, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

KEYS = ['ticker', 'created_time']
TMP_DIR = '/content/moe_tmp'
os.makedirs(TMP_DIR, exist_ok=True)


def normalize_inplace(df, name):
    if df['ticker'].dtype.name != 'category':
        df['ticker'] = df['ticker'].astype('category')
    ct = pd.to_datetime(df['created_time'])
    if ct.dt.tz is not None:
        ct = ct.dt.tz_convert('UTC').dt.tz_localize(None)
    df['created_time'] = ct
    f64 = df.select_dtypes('float64').columns
    if len(f64) > 0:
        df[f64] = df[f64].astype('float32')
    print(f'  {name}: {df.shape}, '
          f'{df.memory_usage(deep=True).sum() / 1e9:.2f} GB')


def softmax_inplace(df, model_name):
    """Convert per-horizon logits to probabilities. Modifies df in place."""
    for h in ['5minutes', '15minutes', '30minutes', '60minutes']:
        cols = [f'probability_{c}_{h}_{model_name}'
                for c in ['no_jump', 'up', 'down']]
        logits = df[cols].to_numpy(dtype=np.float32)
        probs = F.softmax(torch.from_numpy(logits), dim=-1).numpy()
        df[cols] = probs

In [ ]:
print('Stage features...')
df_features = pd.read_parquet(DATA_ROOT + 'all_trades_features.parquet')
normalize_inplace(df_features, 'features')
df_features.sort_values(KEYS, kind='stable', inplace=True)
df_features.reset_index(drop=True, inplace=True)
df_features.to_parquet(f'{TMP_DIR}/features.parquet',
                       engine='pyarrow', compression='zstd', index=False)
print(f'  staged: {os.path.getsize(f"{TMP_DIR}/features.parquet") / 1e9:.2f} GB')
del df_features
gc.collect()

Stage features...
  features: (63801798, 63), 23.38 GB
  staged: 2.69 GB


30

In [ ]:
# Each block: load → preprocess → (softmax if needed) → save preds-only to local disk → free

# ---- lstm_lgbm (row-aligned, save keys + preds) ----
print('\nStage lstm_lgbm...')
df = pd.read_parquet(DATA_ROOT + 'stacking_predictions.parquet')
normalize_inplace(df, 'lstm_lgbm')
df.sort_values(KEYS, kind='stable', inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_parquet(f'{TMP_DIR}/lstm_lgbm.parquet',
              engine='pyarrow', compression='zstd', index=False)
del df; gc.collect()


# ---- fft (row-aligned) ----
print('\nStage fft...')
df = pd.read_parquet(DATA_ROOT + 'ftt_predictions.parquet')
normalize_inplace(df, 'fft')
df.sort_values(KEYS, kind='stable', inplace=True)
df.reset_index(drop=True, inplace=True)
df.to_parquet(f'{TMP_DIR}/fft.parquet',
              engine='pyarrow', compression='zstd', index=False)
del df; gc.collect()


# ---- mamba (logits → softmax → aggregate → save) ----
print('\nStage mamba...')
df = pd.read_parquet(DATA_ROOT + 'mamba_all_trades_logits.parquet')
normalize_inplace(df, 'mamba')
print('  applying softmax to logits...')
softmax_inplace(df, 'mamba')
pred_cols = [c for c in df.columns if c not in KEYS]
before = len(df)
df.dropna(subset=pred_cols, how='any', inplace=True)
print(f'  dropped {before - len(df):,} NaN rows')
print('  aggregating by (ticker, time)...')
df_agg = (df.groupby(KEYS, observed=True, sort=False)[pred_cols]
            .mean()
            .reset_index())
print(f'  aggregated to {len(df_agg):,} unique rows')
df_agg.to_parquet(f'{TMP_DIR}/mamba_agg.parquet',
                  engine='pyarrow', compression='zstd', index=False)
del df, df_agg; gc.collect()


# ---- ctts (probabilities already; aggregate → save) ----
print('\nStage ctts...')
df = pd.read_parquet(DATA_ROOT + 'CTTS_predictions.parquet')
normalize_inplace(df, 'ctts')
pred_cols = [c for c in df.columns if c not in KEYS]
before = len(df)
df.dropna(subset=pred_cols, how='any', inplace=True)
print(f'  dropped {before - len(df):,} NaN rows')
print('  aggregating by (ticker, time)...')
df_agg = (df.groupby(KEYS, observed=True, sort=False)[pred_cols]
            .mean()
            .reset_index())
print(f'  aggregated to {len(df_agg):,} unique rows')
df_agg.to_parquet(f'{TMP_DIR}/ctts_agg.parquet',
                  engine='pyarrow', compression='zstd', index=False)
del df, df_agg; gc.collect()


# ---- moirai (logits → softmax → aggregate → save) ----
print('\nStage moirai...')
df = pd.read_parquet(DATA_ROOT + 'moirai_all_trades_logits.parquet')
normalize_inplace(df, 'moirai')
print('  applying softmax to logits...')
softmax_inplace(df, 'moirai')
pred_cols = [c for c in df.columns if c not in KEYS]
before = len(df)
df.dropna(subset=pred_cols, how='any', inplace=True)
print(f'  dropped {before - len(df):,} NaN rows')
print('  aggregating by (ticker, time)...')
df_agg = (df.groupby(KEYS, observed=True, sort=False)[pred_cols]
            .mean()
            .reset_index())
print(f'  aggregated to {len(df_agg):,} unique rows')
df_agg.to_parquet(f'{TMP_DIR}/moirai_agg.parquet',
                  engine='pyarrow', compression='zstd', index=False)
del df, df_agg; gc.collect()

print('\nAll prediction dfs staged. Local files:')
for f in sorted(os.listdir(TMP_DIR)):
    print(f'  {f:<28}  {os.path.getsize(os.path.join(TMP_DIR, f)) / 1e9:.2f} GB')


Stage lstm_lgbm...
  lstm_lgbm: (63801798, 26), 6.95 GB

Stage fft...
  fft: (63801798, 14), 3.89 GB

Stage mamba...
  mamba: (44267126, 14), 2.66 GB
  applying softmax to logits...
  dropped 0 NaN rows
  aggregating by (ticker, time)...
  aggregated to 32,867,600 unique rows

Stage ctts...
  ctts: (60938701, 14), 3.67 GB
  dropped 5,653,294 NaN rows
  aggregating by (ticker, time)...
  aggregated to 42,854,043 unique rows

Stage moirai...
  moirai: (44267126, 14), 2.66 GB
  applying softmax to logits...
  dropped 0 NaN rows
  aggregating by (ticker, time)...
  aggregated to 32,867,600 unique rows

All prediction dfs staged. Local files:
  ctts_agg.parquet              2.47 GB
  features.parquet              2.69 GB
  fft.parquet                   3.77 GB
  lstm_lgbm.parquet             6.75 GB
  mamba_agg.parquet             1.86 GB
  moirai_agg.parquet            1.84 GB


In [ ]:
# Load features fresh
print('Loading features...')
df_moe = pd.read_parquet(f'{TMP_DIR}/features.parquet')
print(f'  {df_moe.shape}, {df_moe.memory_usage(deep=True).sum() / 1e9:.2f} GB')


# ---- Verified-aligned dfs: load → verify → concat columns → free ----
for name in ['lstm_lgbm', 'fft']:
    print(f'\n{name}: load + verify + concat')
    d = pd.read_parquet(f'{TMP_DIR}/{name}.parquet')

    if len(df_moe) != len(d):
        raise ValueError(f'{name}: length mismatch '
                         f'({len(df_moe)} vs {len(d)})')
    tk_ok = (df_moe['ticker'].values == d['ticker'].values).all()
    ts_ok = (df_moe['created_time'].values == d['created_time'].values).all()
    if not (tk_ok and ts_ok):
        raise ValueError(f'{name}: alignment FAILED '
                         f'(ticker_ok={tk_ok}, time_ok={ts_ok})')
    print(f'  alignment verified on {len(d):,} rows ✓')

    pred_cols = [c for c in d.columns if c not in KEYS]
    df_moe = pd.concat(
        [df_moe.reset_index(drop=True), d[pred_cols].reset_index(drop=True)],
        axis=1,
        copy=False,
    )
    del d; gc.collect()
    print(f'  after concat: {df_moe.shape}, '
          f'{df_moe.memory_usage(deep=True).sum() / 1e9:.2f} GB')


# ---- Aggregated dfs: load → merge → free ----
for name in ['mamba', 'ctts', 'moirai']:
    print(f'\n{name}: load + merge')
    d_agg = pd.read_parquet(f'{TMP_DIR}/{name}_agg.parquet')
    print(f'  loaded {len(d_agg):,} unique (ticker, time) rows')
    df_moe = df_moe.merge(d_agg, on=KEYS, how='left')
    del d_agg; gc.collect()
    print(f'  after merge: {df_moe.shape}, '
          f'{df_moe.memory_usage(deep=True).sum() / 1e9:.2f} GB')


print(f'\n\nFinal: {df_moe.shape}, '
      f'{df_moe.memory_usage(deep=True).sum() / 1e9:.2f} GB')

pred_cols_all = [c for c in df_moe.columns if 'probability' in c]
print(f'\nNaN rates in prediction columns:')
print(df_moe[pred_cols_all].isna().mean().round(3))

Loading features...
  (63801798, 63), 23.38 GB

lstm_lgbm: load + verify + concat
  alignment verified on 63,801,798 rows ✓
  after concat: (63801798, 87), 29.51 GB

fft: load + verify + concat
  alignment verified on 63,801,798 rows ✓
  after concat: (63801798, 99), 32.57 GB

mamba: load + merge
  loaded 32,867,600 unique (ticker, time) rows
  after merge: (63801798, 111), 40.10 GB

ctts: load + merge
  loaded 42,854,043 unique (ticker, time) rows
  after merge: (63801798, 123), 43.17 GB

moirai: load + merge
  loaded 32,867,600 unique (ticker, time) rows
  after merge: (63801798, 135), 46.23 GB


Final: (63801798, 135), 46.23 GB

NaN rates in prediction columns:
probability_down_5minutes_lgbm          0.000
probability_down_5minutes_lstm          0.086
probability_no_jump_5minutes_lgbm       0.000
probability_no_jump_5minutes_lstm       0.086
probability_up_5minutes_lgbm            0.000
                                        ...  
probability_up_30minutes_moirai         0.303
proba

In [ ]:
OUT_PATH = os.path.join(DATA_ROOT, 'moe_data.parquet')
LOCAL_TMP = '/content/moe_data.parquet'

print('Writing locally...')
df_moe.to_parquet(LOCAL_TMP, engine='pyarrow', compression='zstd', index=False)
print(f'Local size: {os.path.getsize(LOCAL_TMP) / 1e9:.2f} GB')

print('Copying to Drive...')
shutil.copy(LOCAL_TMP, OUT_PATH)
print(f'Saved to {OUT_PATH}')

# Optional cleanup: remove the staging directory
shutil.rmtree(TMP_DIR)
print(f'Cleaned up {TMP_DIR}')

Writing locally...
Local size: 18.46 GB
Copying to Drive...
Saved to /content/drive/MyDrive/CS1090B/project/data/moe_data.parquet
Cleaned up /content/moe_tmp


In [ ]:
df = pd.read_parquet(OUT_PATH)
df.head()

,ticker,yes_price,no_price,count,taker_side,created_time,volume,log_volume,target_price_5m,signed_ret_5m,...,probability_down_5minutes_moirai,probability_no_jump_15minutes_moirai,probability_up_15minutes_moirai,probability_down_15minutes_moirai,probability_no_jump_30minutes_moirai,probability_up_30minutes_moirai,probability_down_30minutes_moirai,probability_no_jump_60minutes_moirai,probability_up_60minutes_moirai,probability_down_60minutes_moirai
0,538APPROVEY-24DEC31-T37.9,1,99,304,yes,2025-01-01 02:57:15.947080,304.0,5.720312,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ACPI-24-B1.5,1,99,38,yes,2025-01-01 15:09:30.590342,38.0,3.663562,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ACPI-24-B1.5,1,99,12,yes,2025-01-01 15:09:30.590342,12.0,2.564949,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ACPI-24-B1.5,2,98,42,yes,2025-01-01 15:23:33.219179,84.0,4.442651,1.0,-1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ACPI-24-B1.5,1,99,850,yes,2025-01-01 15:23:33.219179,850.0,6.746412,1.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
